In [1]:
import pandas as pd
import os
import json

In [2]:
df_lm = pd.read_csv('/home/erik/Riksarkivet/Projects/fastighet/data/LM_2008.txt', sep=';', encoding='latin-1', dtype=str, index_col=[3,4,5,6])

In [4]:
%%time
#VÄSTERÅS;*;AKLEJAN;2
#G-VÄXJÖ;ÖSTER;STG;126
"G-VÄXJÖ;VÄXJÖ HOSPITALSGÅRD;3;11",
"G-VÄXJÖ;ÖSTER;STG;126",
"U-VÄSTERÅS;*;ABBORREN;1"
"G-VÄXJÖ;*;BYGGMÄSTAREN;11"
"U-VÄSTERÅS;*;ANKARET;2"
"O-LYSE;ULSERÖD STORA;1;66"
"O-ASKUM;SMÖGEN;SOTEN;6"
"VRETA;1;3""D-NÄSHULTA""D-SUNDBY"
#D-TORSHÄLLA;*;STG;310+/314/
df_test  = df_lm.loc[('G-KJÄTTERYD', '*', 'STG', '401')]
df_test2  = df_lm.loc[('G-VÄXJÖ', 'ÖSTER', 'STG', '13')]

print(df_test['FNR'].iloc[0])
print(df_test2['FNR'].iloc[0])

ipykernel_launcher:12: PerformanceWarning: indexing past lexsort depth may impact performance.


KeyError: 'G-KJÄTTERYD'

In [4]:
with open('/home/erik/Riksarkivet/Projects/fastighet/output/htr_result_on_testset_2.json', 'r') as f:

    eval_dict = json.load(f)

    for j, batch in enumerate(eval_dict.keys()):

        for page in eval_dict[batch].keys():

            if eval_dict[batch][page] is not None and eval_dict[batch][page][0]['det_prob'] is not None:

                for i, inst in enumerate(eval_dict[batch][page]):
                    try:
                        trakt_pred = inst['pred'].split(';')[0]
                        block_pred = inst['pred'].split(';')[1]
                        enhet_pred = inst['pred'].split(';')[2]

                        df_link = df_lm.loc[('ALLE', '1', '1')]

                        eval_dict[batch][page][i]['dir_link'] = len(df_link)


                    except:
                        eval_dict[batch][page][i]['dir_link'] = 0
        print(j)

ipykernel_launcher:17: PerformanceWarning: indexing past lexsort depth may impact performance.


KeyError: 'dir_link'

In [ ]:
for j, batch in enumerate(eval_dict.keys()):

        for page in eval_dict[batch].keys():

            if eval_dict[batch][page] is not None and eval_dict[batch][page][0]['det_prob'] is not None:

                hit = False
                
                for i, inst in enumerate(eval_dict[batch][page]):

                    if eval_dict[batch][page][i]['dir_link'] > 0:

In [5]:
df_batcher1 = pd.read_csv('/home/erik/Riksarkivet/Projects/fastighet/data/batcher_data.txt', '\t', dtype=str, encoding='latin-1')
df_batcher2 = pd.read_csv('/home/erik/Riksarkivet/Projects/fastighet/data/batcher_data2.txt', '\t', dtype=str, encoding='latin-1')
df_batcher = pd.concat([df_batcher1, df_batcher2])

/home/erik/anaconda3/envs/openmmlab/lib/python3.7/site-packages/IPython/core/interactiveshell.py:3444: FutureWarning: In a future version of pandas all arguments of read_csv except for the argument 'filepath_or_buffer' will be keyword-only
  exec(code_obj, self.user_global_ns, self.user_ns)


In [6]:
batches = df_batcher.groupby('batch')

spelling_dict = {}
list_of_batches = df_batcher.batch.unique()
print(len(list_of_batches))

for batch in list_of_batches:
    batch_dict = {}
    
    df_batch = batches.get_group(batch)
    batch_dict['GTRAKT'] = df_batch.GTRAKT.unique()
    batch_dict['GBLOCK'] = df_batch.GBLOCK.unique()
    batch_dict['GENHET'] = df_batch.GENHET.unique()
    batch_dict['GKOMMUN'] = df_batch.GKOMMUN.unique()

    spelling_dict[batch] = batch_dict

217


In [7]:

with open('/home/erik/Riksarkivet/Projects/fastighet/output/link_res_testset.json', 'r') as f:

    links_df = json.load(f) 

    for i, batch in enumerate(links_df.keys()):
        try:
            gkomm = spelling_dict[batch]['GKOMMUN']
        except:
            continue

        print(gkomm)

        no_of_non_disamb = 0
        no_of_disamb = 0
        no_of_misses = 0
        no_of_ambiguities = 0

        for page in links_df[batch].keys():

            if links_df[batch][page] is not None and links_df[batch][page][0]['det_prob'] is not None:

                for inst in links_df[batch][page]:


                    if 'dir_link' in inst and inst['dir_link'] > 0:

                        trakt = inst['pred'].split(';')[0].upper()
                        block = inst['pred'].split(';')[1].upper()
                        enhet = inst['pred'].split(';')[2].upper()
                        
            
                        no_of_val_hits = 0
                        val_gks = []
                        for gk in gkomm:
                            
                            try:
                                df_test  = df_lm.loc[(gk, trakt, block, enhet)]
                            except:
                                continue
                            
                            if len(df_test) > 1:
                                no_of_non_disamb += 1
                            elif len(df_test) == 1:

                                inst['validated_pred'] = trakt + ';' + block + ';' + enhet
                                val_gks.append(gk)
                                no_of_val_hits += 1
                            else:
                                no_of_misses += 1

                        inst['val_gks'] = val_gks

                        

        print(str(i) + ':' + batch + ':' + str(no_of_ambiguities))

['G-VÄXJÖ']
ipykernel_launcher:37: PerformanceWarning: indexing past lexsort depth may impact performance.
0:10001009:0
['G-VÄXJÖ']
1:10001059:0
['G-SKATELÖV']
2:10001239:0
['G-ÅSEDA']
3:10001419:0
['G-ÄLGHULT']
4:10001429:0
['Y-NJURUNDA']
5:10001490:0
['Y-NJURUNDA']
6:10001510:0
['Y-STÖDE']
7:10001722:0
['Y-LJUSTORP']
8:10001933:0
['O-GÖTEBORG']
9:10002043:0
['O-GÖTEBORG']
10:10002390:0
['O-GÖTEBORG']
11:10002455:0
['O-PARTILLE']
13:10003246:0
['O-PARTILLE']
14:10003296:0
['O-MÖLNDAL' 'O-FÄSSBERG']
15:10003466:0
['O-UDDEVALLA']
16:10003654:0
['O-UDDEVALLA']
17:10003664:0
['O-LANE-RYR']
18:10003724:0
['O-LYSEKIL']
19:10003774:0
['O-SKAFTÖ']
20:10003814:0
['O-ASKUM']
21:10003833:0
['E-NORRKÖPING']
23:10004019:0
['E-HÄLLESTAD']
24:10004241:0
['F-NORRA HESTRA']
29:10004952:0
['F-SKILLINGARYD']
30:10004992:0
['G-TRARYD']
31:10005266:0
['G-VRÅ' 'G-LJUNGBY' 'G-AGUNNARYD' 'G-ANNERSTAD' 'G-TUTARYD' 'G-ANGELSTAD'
 'G-RYSSBY' 'G-BOLMSÖ' 'G-NÖTTJA' 'G-TORPA' 'G-LIDHULT' 'G-BERGA'
 'G-KÅNNA' 'G-OD

In [8]:
with open('/home/erik/Riksarkivet/Projects/fastighet/output/validated_testset.json', 'w', encoding='utf8') as f:
    j = json.dumps(links_df, indent = 4, ensure_ascii=False)
    f.write(str(j))